# ML-05 — Ranking Signal Feature Vector and Leakage Check

This notebook continues the ML-04 Ranking Signal Analysis contract on the March 2026 warehouse slice. It builds the same five first-half features, keeps the second-half movement proxy separate, and attacks the feature vector with a deliberate label leak.

## 1. Build the Ranking Signal feature vector

The decision moment is March 15, 2026. Features use only March 1–15 daily GSC observations. The label proxy uses the later March 16–31 window.

In [2]:
import getpass
import os

import duckdb
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

MONTH = "2026-03"
MIDPOINT = "2026-03-15"
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Enter your Hugging Face READ token (input is hidden): "
)
assert HF_TOKEN, "A Hugging Face READ token is required; it is never stored in this notebook."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

feature_frame = con.sql(f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
            SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
            AVG(CASE WHEN report_date <= DATE '{MIDPOINT}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS first_half_avg_position,
            COUNT(DISTINCT CASE WHEN report_date <= DATE '{MIDPOINT}' AND gsc_impressions > 0 THEN report_date END) AS first_half_active_days,
            SUM(CASE WHEN report_date > DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END) AS second_half_impressions,
            COUNT(DISTINCT CASE WHEN report_date > DATE '{MIDPOINT}' THEN report_date END) AS second_half_days,
            COUNT(DISTINCT CASE WHEN report_date <= DATE '{MIDPOINT}' THEN report_date END) AS first_half_days
        FROM {FACT_MONTH}
        GROUP BY 1, 2
    )
    SELECT
        client_hash_id,
        content_hash_id,
        first_half_impressions,
        first_half_clicks,
        100.0 * first_half_clicks / NULLIF(first_half_impressions, 0) AS first_half_ctr,
        COALESCE(first_half_avg_position, 0) AS first_half_avg_position,
        first_half_active_days,
        CAST(
            second_half_impressions / NULLIF(second_half_days, 0)
            < 0.8 * first_half_impressions / NULLIF(first_half_days, 0)
            AS INTEGER
        ) AS declined_second_half
    FROM monthly
    WHERE first_half_impressions > 0
      AND second_half_days > 0
""").df()

feature_columns = [
    "first_half_impressions",
    "first_half_clicks",
    "first_half_ctr",
    "first_half_avg_position",
    "first_half_active_days",
]
X = feature_frame[feature_columns].copy()
y = feature_frame["declined_second_half"].astype(int)
groups = feature_frame["client_hash_id"]

assert len(feature_columns) == 5
assert X.notna().all().all()
assert y.nunique() == 2
assert groups.nunique() > 1

print(f"Warehouse month: {MONTH}")
print(f"Feature rows with complete two-half outcomes: {len(X):,}")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")
print(f"Clients retained for grouped validation: {groups.nunique():,}")
print(f"Observed decline-proxy rate: {y.mean():.3f}")

Warehouse month: 2026-03
Feature rows with complete two-half outcomes: 151,980
Feature columns (5): ['first_half_impressions', 'first_half_clicks', 'first_half_ctr', 'first_half_avg_position', 'first_half_active_days']
Clients retained for grouped validation: 44
Observed decline-proxy rate: 0.359


## 2. Feature notes: meaning and available-when?

- `first_half_impressions`: available from Search Console through March 15.
- `first_half_clicks`: available from Search Console through March 15.
- `first_half_ctr`: calculated from the two first-half counts, so no later information is used.
- `first_half_avg_position`: available from first-half position observations; `0` means no position data.
- `first_half_active_days`: counted from first-half daily rows with impressions.

The identifiers are retained only as context for grouped validation. The second-half proxy label is never part of the honest feature matrix.

In [3]:
feature_notes = pd.DataFrame([
    {
        "feature": "first_half_impressions",
        "meaning": "GSC impressions in March 1-15",
        "available_when": "at the March 15 decision moment",
        "handling": "non-negative warehouse count",
    },
    {
        "feature": "first_half_clicks",
        "meaning": "GSC clicks in March 1-15",
        "available_when": "at the March 15 decision moment",
        "handling": "non-negative warehouse count",
    },
    {
        "feature": "first_half_ctr",
        "meaning": "first-half clicks divided by impressions",
        "available_when": "at the March 15 decision moment",
        "handling": "derived only from first-half fields",
    },
    {
        "feature": "first_half_avg_position",
        "meaning": "mean positive GSC position in March 1-15",
        "available_when": "at the March 15 decision moment",
        "handling": "zero sentinel means no position data",
    },
    {
        "feature": "first_half_active_days",
        "meaning": "first-half dates with impressions",
        "available_when": "at the March 15 decision moment",
        "handling": "daily count",
    },
])
print(feature_notes.to_string(index=False))
print(f"Missing values in X: {int(X.isna().sum().sum())}")
assert len(feature_notes) == 5
assert X.isna().sum().sum() == 0

                feature                                  meaning                  available_when                             handling
 first_half_impressions            GSC impressions in March 1-15 at the March 15 decision moment         non-negative warehouse count
      first_half_clicks                 GSC clicks in March 1-15 at the March 15 decision moment         non-negative warehouse count
         first_half_ctr first-half clicks divided by impressions at the March 15 decision moment  derived only from first-half fields
first_half_avg_position mean positive GSC position in March 1-15 at the March 15 decision moment zero sentinel means no position data
 first_half_active_days        first-half dates with impressions at the March 15 decision moment                          daily count
Missing values in X: 0


## 3. Attack the vector: label, time, identifiers, and grouped validation

The strongest test is intentional: add an exact copy of the second-half label, score it, then remove it. The honest score uses a client-grouped split so content from the same client does not appear in both train and test.

In [4]:
label_derived = {"declined_second_half", "leak_label_copy"}
outcome_window_fields = {
    "second_half_impressions", "second_half_days", "declined_second_half",
}
identifiers = {"client_hash_id", "content_hash_id"}

honest_source_columns = set(feature_columns)
print(f"Label-derived fields in honest X: {sorted(honest_source_columns & label_derived)}")
print(f"Outcome-window fields in honest X: {sorted(honest_source_columns & outcome_window_fields)}")
print(f"Identifiers in honest X: {sorted(honest_source_columns & identifiers)}")
assert not honest_source_columns & label_derived
assert not honest_source_columns & outcome_window_fields
assert not honest_source_columns & identifiers

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_positions, test_positions = next(splitter.split(X, y, groups=groups))

leaky_X = X.copy()
leaky_X["leak_label_copy"] = y.to_numpy()


def grouped_score(frame):
    model = DecisionTreeClassifier(max_depth=3, random_state=42)
    model.fit(frame.iloc[train_positions], y.iloc[train_positions])
    return model.score(frame.iloc[test_positions], y.iloc[test_positions])

majority_baseline = max(
    y.iloc[test_positions].mean(),
    1 - y.iloc[test_positions].mean(),
)
leaky_score = grouped_score(leaky_X)
honest_score = grouped_score(X)

print(f"Client-grouped test clients: {groups.iloc[test_positions].nunique():,}")
print(f"Majority baseline: {majority_baseline:.3f}")
print(f"Score with deliberate label leak: {leaky_score:.3f}")
print(f"Honest grouped score after leak removal: {honest_score:.3f}")

assert leaky_score >= 0.99
assert "leak_label_copy" not in X.columns
assert honest_score < leaky_score
print("Leakage attack passed: the score jumps when the answer is supplied and falls after removal.")

Label-derived fields in honest X: []
Outcome-window fields in honest X: []
Identifiers in honest X: []
Client-grouped test clients: 11
Majority baseline: 0.607
Score with deliberate label leak: 1.000
Honest grouped score after leak removal: 0.604
Leakage attack passed: the score jumps when the answer is supplied and falls after removal.


## 4. What is excluded and why

- `declined_second_half`: the observed outcome proxy; using it as input reveals the answer.
- `second_half_impressions` and related second-half aggregates: they occur after the March 15 decision moment.
- `client_hash_id` and `content_hash_id`: pseudonymous identifiers used only for grouped validation and audit.
- Any product score or action flag: it would reproduce an existing decision rather than test independent ranking signals.
- Raw queries, URLs, client names, and tokens: private or unnecessary for the public-safe analysis.

The final `X` contains only the five first-half features.

In [5]:
final_feature_columns = set(X.columns)
print(f"Final feature count: {len(final_feature_columns)}")
print(f"Final feature columns: {sorted(final_feature_columns)}")
print("Validation design: client_hash_id remains outside X for grouped holdout only.")

assert final_feature_columns == set(feature_columns)
assert not final_feature_columns & label_derived
assert not final_feature_columns & outcome_window_fields
assert not final_feature_columns & identifiers
assert "client_hash_id" not in X.columns
assert "content_hash_id" not in X.columns
print("Final exclusion audit passed.")

Final feature count: 5
Final feature columns: ['first_half_active_days', 'first_half_avg_position', 'first_half_clicks', 'first_half_ctr', 'first_half_impressions']
Validation design: client_hash_id remains outside X for grouped holdout only.
Final exclusion audit passed.


## Self-check

- [x] Five Ranking Signal features are built from the March first-half warehouse slice.
- [x] Every feature has an explicit available-when explanation.
- [x] The label-derived leak is added, scored, removed, and tested again.
- [x] The honest score uses a client-grouped split and prints the majority baseline.
- [x] Identifiers and later outcome fields are excluded from `X`.
- [x] No raw identifiers, queries, URLs, client names, or tokens are printed.
- [ ] Run all cells with warehouse access, commit the executed notebook, and submit the repository URL.